# Bakehouse Customer & Transactions

**Dataset:** `samples.bakehouse.sales_customers`, `samples.bakehouse.sales_transactions`

**Difficulty:** Medium

**Topics:** joins, aggregation, window, anti-join

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

customers = spark.read.table("samples.bakehouse.sales_customers")
transactions = spark.read.table("samples.bakehouse.sales_transactions")

In [0]:
customers.printSchema()
transactions.printSchema()

## Problem 1

Join customers to transactions on `customerID`. Count transactions and total spend per customer country.

**Expected output columns:**
- `country`
- `transaction_count`
- `total_spend`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = transactions.join(customers, "customerID", "inner").groupBy("country").agg(
    F.count("*").alias("transaction_count"),
    F.sum("totalPrice").alias("total_spend")
)
result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'total_spend' in cols, "Missing column: total_spend"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_spend = result_1.agg(F.min('total_spend')).collect()[0][0]
assert min_spend >= 0, f"Expected total_spend >= 0, got min={min_spend}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find the top 10 customers by total spend. Join customers and transactions.

**Expected output columns:**
- `customerID`
- `first_name`
- `last_name`
- `country`
- `total_spend`

In [0]:
result_2 = transactions.select(
    "customerID", "totalPrice"
).join(
    F.broadcast(customers.select("customerID", "first_name", "last_name", "country")),
    "customerID"
).groupBy("customerID", "first_name", "last_name", "country").agg(
    F.sum("totalPrice").alias("total_spend")
).orderBy(F.col("total_spend").desc()).limit(10)

In [0]:
result_2.display()

In [0]:
customer_spend = transactions.groupBy("customerID").agg(
    F.sum("totalPrice").alias("total_spend")
)
result_2 = customers.join(customer_spend, "customerID").select(
    "customerID", "first_name", "last_name", "country", "total_spend"
).orderBy(F.col("total_spend").desc()).limit(10)

result_2.display()

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

top_customers = transactions.groupBy("customerID").agg(
    F.sum("totalPrice").alias("total_spend")
).orderBy(F.col("total_spend").desc()).limit(10)

result_2 = customers.join(F.broadcast(top_customers), "customerID").select(
    "customerID", "first_name", "last_name", "country", "total_spend"
)

result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'first_name' in cols, "Missing column: first_name"
assert 'last_name' in cols, "Missing column: last_name"
assert 'country' in cols, "Missing column: country"
assert 'total_spend' in cols, "Missing column: total_spend"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 rows (top 10), got {cnt}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Find customers who have never made a transaction using a left anti-join (or left join + null filter).

**Expected output columns:**
- `customerID`
- `first_name`
- `last_name`
- `country`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = F.broadcast(customers).join(transactions, "customerID", "anti").select(
    "customerID", "first_name", "last_name", "country"
)
result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'first_name' in cols, "Missing column: first_name"
assert 'last_name' in cols, "Missing column: last_name"
assert 'country' in cols, "Missing column: country"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
# It is valid for there to be 0 customers without transactions, but the DataFrame must exist
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Calculate the average basket size - the average number of unique products per customer per transaction - per country.

**Expected output columns:**
- `country`
- `avg_basket_size`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = transactions.join(F.broadcast(customers), "customerID").groupBy(
    "country", "transactionID"
).agg(
    F.count("*").alias("basket_size")
).groupBy("country").agg(
    F.avg("basket_size").alias("avg_basket_size")
)

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'country' in cols, "Missing column: country"
assert 'avg_basket_size' in cols, "Missing column: avg_basket_size"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_basket = result_4.agg(F.min('avg_basket_size')).collect()[0][0]
assert min_basket >= 1, f"Expected avg_basket_size >= 1, got min={min_basket}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

For each customer find their first purchase date and last purchase date.

**Expected output columns:**
- `customerID`
- `first_name`
- `last_name`
- `first_purchase`
- `last_purchase`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = transactions.join(F.broadcast(customers), "CustomerID").groupBy(
    "customerID","first_name", "last_name"
).agg(
    F.min("dateTime").alias("first_purchase"),
    F.max("dateTime").alias("last_purchase")
)

result_5.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'first_name' in cols, "Missing column: first_name"
assert 'last_name' in cols, "Missing column: last_name"
assert 'first_purchase' in cols, "Missing column: first_purchase"
assert 'last_purchase' in cols, "Missing column: last_purchase"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
invalid = result_5.filter(F.col('first_purchase') > F.col('last_purchase')).count()
assert invalid == 0, f"Found {invalid} rows where first_purchase > last_purchase"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Find customers who bought the same product more than once. Show the purchase count per customer-product pair. Filter where `purchase_count > 1`.

**Expected output columns:**
- `customerID`
- `product`
- `purchase_count`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
result_6 = transactions.groupBy("customerID", "product").agg(
    F.count("*").alias("purchase_count")
).filter(F.col("purchase_count") > 1)

result_6.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'product' in cols, "Missing column: product"
assert 'purchase_count' in cols, "Missing column: purchase_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_pc = result_6.agg(F.min('purchase_count')).collect()[0][0]
assert min_pc > 1, f"Expected purchase_count > 1 for all rows, found min={min_pc}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

For each continent compute revenue and percentage share of total revenue.

**Expected output columns:**
- `continent`
- `total_revenue`
- `revenue_pct`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_7 = transactions.join(F.broadcast(customers), "customerID").groupBy("continent").agg(
    F.sum("totalPrice").alias("total_revenue")
).withColumn(
    "revenue_pct",
    (100*F.col("total_revenue")/F.sum("total_revenue").over(w))
)

result_7.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'continent' in cols, "Missing column: continent"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'revenue_pct' in cols, "Missing column: revenue_pct"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_pct = result_7.agg(F.max('revenue_pct')).collect()[0][0]
min_pct = result_7.agg(F.min('revenue_pct')).collect()[0][0]
assert max_pct <= 100, f"Expected revenue_pct <= 100, got max={max_pct}"
assert min_pct >= 0, f"Expected revenue_pct >= 0, got min={min_pct}"
print(f"Problem 7 passed ✓  ({cnt} rows)")